<a href="https://colab.research.google.com/github/HiveCase/DLP-DA5013/blob/main/Deep_Learning_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## L8: Building & training a tokenizer: byte-pair encoding with bookcorpus

In [3]:
# !pip install -U "datasets==3.6.0"

In [4]:
from datasets import load_dataset

In [1]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import Lowercase

In [7]:
from pprint import pprint

In [8]:
ds = load_dataset('bookcorpus/bookcorpus', split='all')
pprint(ds)

Dataset({
    features: ['text'],
    num_rows: 74004228
})


In [9]:
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
tokenizer.normalizer = Lowercase()

In [10]:
from tokenizers.trainers import BpeTrainer

In [11]:
trainer = BpeTrainer(vocab_size=32000, special_tokens=["[PAD]","[UNK]"], continuing_subword_prefix="##")

In [12]:
# batch tokenization function
def get_examples(batch_size=1000):
  for i in range(0, len(ds), batch_size):
    yield ds[i:i+batch_size]['text']

In [13]:
from multiprocessing import cpu_count

In [14]:
print(cpu_count())

2


In [15]:
tokenizer.train_from_iterator(get_examples(batch_size=10000), trainer=trainer, length=len(ds))

In [19]:
import os

os.makedirs("model", exist_ok=True)

tokenizer.model.save("model", prefix="hopper")

['model/hopper-vocab.json', 'model/hopper-merges.txt']

In [20]:
with open('model/hopper-merges.txt', 'r') as file:
  row = 0
  num_lines = 10
  for line in file.readlines():
    print(line)
    row+=1
    if row>=num_lines:
      break

#version: 0.2

##h ##e

t ##he

##i ##n

##e ##r

##e ##d

##o ##u

##n ##d

##in ##g

t ##o



In [21]:
with open('model/hopper-merges.txt', 'r') as file:
  row = 0
  num_lines = 10
  for line in reversed(file.readlines()):
    print(line)
    row+=1
    if row>=num_lines:
      break

mel ##anthe

black ##er

ad ##ject

v ##ang

betroth ##al

tiptoe ##ing

restroom ##s

consol ##ing

esp ##ionage

influ ##x



In [22]:
with open('model/hopper-merges.txt', 'r') as file:
  lines = file.readlines()
print(f"Number of merges: {len(lines)}")

Number of merges: 31871


## L9: Encdode & decoder | tokenization explained with BERT, padding & truncation

In [23]:
sample = ds[0]['text']
print(f'Sample: {sample}')
encoding = tokenizer.encode(sample)
print(encoding)

Sample: usually , he would be tearing around the living room , playing with his toys .
Encoding(num_tokens=16, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])


In [24]:
tokenizer.save('hopper.json')

In [26]:
import json
with open('hopper.json', 'r') as file:
  json_data = json.load(file)

In [28]:
pprint(json_data, depth =1)

{'added_tokens': [...],
 'decoder': None,
 'model': {...},
 'normalizer': {...},
 'padding': None,
 'post_processor': None,
 'pre_tokenizer': {...},
 'truncation': None,
 'version': '1.0'}


In [30]:
# loading tokenizer from file
trained_tokenizer = Tokenizer(BPE())
trained_tokenizer = trained_tokenizer.from_file('hopper.json')
tokens = trained_tokenizer.encode(sample).tokens
print(tokens)

['usually', ',', 'he', 'would', 'be', 'tearing', 'around', 'the', 'living', 'room', ',', 'playing', 'with', 'his', 'toys', '.']


In [31]:
# BERT Tokenizer
bert_tokenizer = Tokenizer(BPE(unk_token='[UNK]'))
bert_tokenizer.normalizer = Lowercase()
bert_tokenizer.pre_tokenizer = Whitespace()
bert_trainer = BpeTrainer(vocab_size=32000,
                          special_tokens=["[PAD]", "[UNK]", "[SEP]", "[MASK]"],
                          continuing_subword_prefix='##')

In [32]:
from tokenizers.processors import TemplateProcessing
bert_tokenizer.post_processor = TemplateProcessing(single="[CLS] $0 [SEP]",
                                                   pair="[CLS] $A [SEP] $B:1",
                                                   special_tokens=[("[CLS]", 2), ("[SEP]", 3)]

)

In [33]:
bert_tokenizer.train_from_iterator(get_examples(batch_size=10000), trainer=bert_trainer, length=len(ds))

In [34]:
text = "All these are so simple to do in HF. Let's do more"
encoded = bert_tokenizer.encode(text)
tokens = encoded.tokens
ids = encoded.ids
out_dict = {'tokens':tokens, 'ids':ids}
pprint(out_dict, depth=2, compact=True)

{'ids': [2, 269, 955, 335, 230, 2533, 140, 205, 156, 55, 96, 23, 461, 16, 66,
         205, 386, 3],
 'tokens': ['[CLS]', 'all', 'these', 'are', 'so', 'simple', 'to', 'do', 'in',
            'h', '##f', '.', 'let', "'", 's', 'do', 'more', '[SEP]']}


In [35]:
text = "All these are so simple to do in HF. Let's do more"
pair = "We have along way to go!"
encoded = bert_tokenizer.encode(text, pair)
tokens = encoded.tokens
ids = encoded.ids
out_dict = {'tokens':tokens, 'ids':ids}
pprint(out_dict, depth=2, compact=True)

{'ids': [2, 269, 955, 335, 230, 2533, 140, 205, 156, 55, 96, 23, 461, 16, 66,
         205, 386, 3, 213, 249, 1052, 414, 140, 259, 11],
 'tokens': ['[CLS]', 'all', 'these', 'are', 'so', 'simple', 'to', 'do', 'in',
            'h', '##f', '.', 'let', "'", 's', 'do', 'more', '[SEP]', 'we',
            'have', 'along', 'way', 'to', 'go', '!']}


In [36]:
plain_tokens = bert_tokenizer.decode(ids)
print(plain_tokens)

all these are so simple to do in h ##f . let ' s do more we have along way to go !


In [43]:
from tokenizers.decoders import WordPiece

In [44]:
bert_tokenizer.decoder = WordPiece(prefix='##')

In [46]:
plain_tokens = bert_tokenizer.decode(ids)
print(plain_tokens)

TypeError: 'tokenizers.decoders.WordPiece' object is not callable

## L10: Wrap it with pre trained tokenizer

In [47]:
from transformers import PreTrainedTokenizerFast

In [48]:
pt_tokenizer = PreTrainedTokenizerFast(tokenizer_file='hopper.json',
                                       unk_token='[UNK]',
                                       pad_token='[PAD]',
                                       model_input_names=["input_ids", "token_type_ids","attention_mask"],
                                       )

In [49]:
model_inputs = pt_tokenizer(text)
pprint(model_inputs, compact=True)

{'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'input_ids': [267, 953, 333, 228, 2531, 138, 203, 154, 53, 80, 21, 459, 14, 64,
               203, 384],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [50]:
model_inputs = pt_tokenizer(text, text_pair=pair)
pprint(model_inputs, compact=True)

{'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                    1, 1, 1],
 'input_ids': [267, 953, 333, 228, 2531, 138, 203, 154, 53, 80, 21, 459, 14, 64,
               203, 384, 211, 247, 1050, 412, 138, 257, 9],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1,
                    1, 1, 1]}


In [51]:
batch_text = [
    'I like the book The Psychology of Money',
    "I enjoyed watching the Transformers movie",
    'oh! thanks for this'
]

In [52]:
model_inputs = pt_tokenizer(batch_text)
pprint(model_inputs, compact=True)

{'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1],
                    [1, 1, 1, 1, 1]],
 'input_ids': [[54, 281, 131, 1701, 131, 19478, 153, 1564],
               [54, 4096, 1443, 131, 7744, 307, 3760],
               [772, 9, 1767, 200, 254]],
 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0],
                    [0, 0, 0, 0, 0]]}


In [53]:
model_inputs = pt_tokenizer(batch_text, padding=True)
pprint(model_inputs, compact=True)

{'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 0],
                    [1, 1, 1, 1, 1, 0, 0, 0]],
 'input_ids': [[54, 281, 131, 1701, 131, 19478, 153, 1564],
               [54, 4096, 1443, 131, 7744, 307, 3760, 0],
               [772, 9, 1767, 200, 254, 0, 0, 0]],
 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0],
                    [0, 0, 0, 0, 0, 0, 0, 0]]}
